# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, cross_val_score, ParameterGrid
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tqdm.notebook import tqdm
import joblib
import itertools
from sklearn.pipeline import Pipeline

## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [2]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
    """Extracts hour and weekday from timestamp, drops timestamp."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X['timestamp'] = pd.to_datetime(X['timestamp'])
        X['hour'] = X['timestamp'].dt.hour
        X['dayofweek'] = X['timestamp'].dt.dayofweek
        X = X.drop(columns=['timestamp'])
        return X

In [3]:
class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    """
    One‑hot encodes categorical features, keeps the target column.
    Returns a single DataFrame with features and the target column.
    """
    def __init__(self, target_col):
        self.target_col = target_col
        self.encoder = OneHotEncoder(sparse=False, handle_unknown='ignore')
        self.cat_features_ = None

    def fit(self, X, y=None):
        # Separate target from features if present
        if self.target_col in X.columns:
            X_ = X.drop(columns=[self.target_col])
            self.y_ = X[self.target_col]
        else:
            X_ = X
            self.y_ = None

        # Identify categorical columns (object dtype)
        self.cat_features_ = X_.select_dtypes(include=['object']).columns.tolist()
        if self.cat_features_:
            self.encoder.fit(X_[self.cat_features_])
        return self

    def transform(self, X):
        X = X.copy()
        # Separate target from features
        if self.target_col in X.columns:
            y = X[self.target_col]
            X_ = X.drop(columns=[self.target_col])
        else:
            X_ = X
            y = None

        # One‑hot encode categorical columns
        if self.cat_features_:
            encoded = self.encoder.transform(X_[self.cat_features_])
            # In scikit-learn 0.23.1, use get_feature_names() with input_features
            feature_names = self.encoder.get_feature_names(input_features=self.cat_features_)
            encoded_df = pd.DataFrame(encoded, columns=feature_names, index=X_.index)
            # Drop original categorical columns and concatenate
            X_ = X_.drop(columns=self.cat_features_)
            X_ = pd.concat([X_, encoded_df], axis=1)

        # Add target back if it was present
        if y is not None:
            X_[self.target_col] = y

        return X_

In [4]:
class TrainValidationTest:
    """Splits data into train, validation, and test sets (stratified)."""
    def __init__(self, test_size=0.2, val_size=0.3, random_state=21):
        self.test_size = test_size
        self.val_size = val_size
        self.random_state = random_state

    def split(self, X, y):
        # First split: train+val vs test
        X_train_val, X_test, y_train_val, y_test = train_test_split(
            X, y, test_size=self.test_size, random_state=self.random_state, stratify=y
        )
        # Then split train+val into train and validation
        X_train, X_valid, y_train, y_valid = train_test_split(
            X_train_val, y_train_val, test_size=self.val_size,
            random_state=self.random_state, stratify=y_train_val
        )
        return X_train, X_valid, X_test, y_train, y_valid, y_test

## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [5]:
class ModelSelection:
    """
    Performs manual grid search over multiple model classes,
    selects the best model based on validation accuracy.
    """
    def __init__(self, models_list):
        """
        models_list : list of tuples (model_class, param_grid, name)
        e.g. (SVC, param_grid_svm, 'SVM')
        """
        self.models_list = models_list
        self.best_models_info = []   # store (name, best_params, best_valid_score)
        self.best_model_name = None
        self.best_model = None

    def choose(self, X_train, y_train, X_valid, y_valid):
        """
        Iterates over each model class, runs grid search with cross‑validation,
        selects best parameters for each class, then evaluates on validation set.
        Returns the name of the best overall model.
        """
        best_overall_score = -1
        best_overall_name = None

        for model_class, param_grid, name in self.models_list:
            print(f"\nEstimator: {name}")
            best_params = None
            best_cv_score = -1
            best_cv_std = None

            # Generate all parameter combinations
            param_combos = list(ParameterGrid(param_grid))
            # Use tqdm to show progress
            for params in tqdm(param_combos, desc=f"{name} grid"):
                # Instantiate model
                model = model_class(**params)
                # Perform cross‑validation
                scores = cross_val_score(model, X_train, y_train, cv=5, n_jobs=-1)
                mean_score = scores.mean()
                if mean_score > best_cv_score:
                    best_cv_score = mean_score
                    best_cv_std = scores.std()
                    best_params = params

            # Now train the best model on the full training set
            best_model = model_class(**best_params)
            best_model.fit(X_train, y_train)
            y_pred_valid = best_model.predict(X_valid)
            valid_score = accuracy_score(y_valid, y_pred_valid)

            print(f"Best params: {best_params}")
            print(f"Best training accuracy (cv mean): {best_cv_score:.3f}")
            print(f"Validation set accuracy score for best params: {valid_score:.3f}")

            # Store info for later
            self.best_models_info.append({
                'model': name,
                'params': best_params,
                'valid_score': valid_score
            })

            # Update overall best
            if valid_score > best_overall_score:
                best_overall_score = valid_score
                best_overall_name = name
                self.best_model = best_model

        self.best_model_name = best_overall_name
        print(f"\nClassifier with best validation set accuracy: {best_overall_name}")
        return best_overall_name

    def best_results(self):
        """Returns a DataFrame with the best model info for each model class."""
        df = pd.DataFrame(self.best_models_info)
        return df[['model', 'params', 'valid_score']]

## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [6]:
class Finalize:
    """Trains the best model on the full training set and evaluates on test set."""
    def __init__(self, estimator):
        self.estimator = estimator

    def final_score(self, X_train, y_train, X_test, y_test):
        """Fits the estimator, predicts on test, prints and returns accuracy."""
        self.estimator.fit(X_train, y_train)
        y_pred = self.estimator.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        print(f"Accuracy of the final model is {acc:.6f}")
        return acc

    def save_model(self, path):
        """Saves the estimator to the given path."""
        joblib.dump(self.estimator, path)
        print(f"Model saved to {path}")

## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [7]:
df = pd.read_csv('../data/checker_submits.csv')

In [8]:
preprocessing = Pipeline([
        ('feature_extractor', FeatureExtractor()),
        ('onehot_encoder', MyOneHotEncoder(target_col='dayofweek'))
])


In [9]:
data = preprocessing.fit_transform(df)

In [10]:
X = data.drop(columns=['dayofweek'])
y = data['dayofweek']

In [11]:
splitter = TrainValidationTest(test_size=0.2, val_size=0.3, random_state=21)
X_train, X_valid, X_test, y_train, y_valid, y_test = splitter.split(X, y)

In [12]:
svm_param_grid = {
    'kernel': ['linear', 'rbf', 'sigmoid'],
    'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': [None, 'balanced'],
    'random_state': [21],
    'probability': [True]
}

In [13]:
tree_param_grid = {
    'max_depth': list(range(1, 50)),
    'class_weight': [None, 'balanced'],
    'criterion': ['gini', 'entropy'],
    'random_state': [21]
}

In [14]:
rf_param_grid = {
    'n_estimators': [5, 10, 50, 100],
    'max_depth': list(range(1, 50)),
    'class_weight': [None, 'balanced'],
    'criterion': ['gini', 'entropy'],
    'random_state': [21]
}

In [15]:
models_list = [
    (SVC, svm_param_grid, 'SVM'),
    (DecisionTreeClassifier, tree_param_grid, 'Decision Tree'),
    (RandomForestClassifier, rf_param_grid, 'Random Forest')
]

In [16]:
selector = ModelSelection(models_list)
best_name = selector.choose(X_train, y_train, X_valid, y_valid)


Estimator: SVM



Best params: {'C': 10, 'class_weight': 'balanced', 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy (cv mean): 0.837
Validation set accuracy score for best params: 0.812

Estimator: Decision Tree



Best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': 22, 'random_state': 21}
Best training accuracy (cv mean): 0.847
Validation set accuracy score for best params: 0.842

Estimator: Random Forest



Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 20, 'n_estimators': 50, 'random_state': 21}
Best training accuracy (cv mean): 0.897
Validation set accuracy score for best params: 0.862

Classifier with best validation set accuracy: Random Forest


In [17]:
results_df = selector.best_results()
print("\nBest results per model:")
print(results_df)


Best results per model:
           model                                             params  \
0            SVM  {'C': 10, 'class_weight': 'balanced', 'gamma':...   
1  Decision Tree  {'class_weight': None, 'criterion': 'gini', 'm...   
2  Random Forest  {'class_weight': None, 'criterion': 'entropy',...   

   valid_score  
0     0.812346  
1     0.841975  
2     0.861728  


In [18]:
final = Finalize(selector.best_model)
final_acc = final.final_score(X_train, y_train, X_test, y_test)

Accuracy of the final model is 0.890533


In [19]:
filename = f"../data/{best_name.replace(' ', '_')}_{final_acc:.6f}.sav"
final.save_model(filename)

Model saved to ../data/Random_Forest_0.890533.sav


In [20]:
loaded_model = joblib.load(filename)
final2 = Finalize(loaded_model)
loaded_acc = final2.final_score(X_train, y_train, X_test, y_test)
assert np.isclose(loaded_acc, final_acc), "Loaded model accuracy differs"
print("Loaded model produces the same accuracy as before.")

Accuracy of the final model is 0.890533
Loaded model produces the same accuracy as before.
